# 📦 Notebook 1: Cài đặt & Khám phá Dataset (EDA)

**Mục tiêu:**
- Cài đặt thư viện cần thiết
- Tải dataset MovieLens 1M
- Khám phá cấu trúc dữ liệu
- Hiểu phân bố ratings, users, movies

**Dataset:** MovieLens 1M — 1 triệu ratings, 6,040 users, 3,706 phim

## 1. Cài đặt thư viện

In [ ]:
# Cài đặt thư viện cần thiết
!pip install scikit-surprise scikit-learn pandas numpy matplotlib seaborn -q

## 2. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('✅ Import thành công!')

## 3. Tải Dataset

In [ ]:
# Download MovieLens 1M
data_dir = Path('data/raw/ml-1m')
if not data_dir.exists():
    !mkdir -p data/raw
    !wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O /tmp/ml-1m.zip
    !unzip -q /tmp/ml-1m.zip -d data/raw/
    print('✅ Dataset đã được tải!')
else:
    print('✅ Dataset đã tồn tại!')

## 4. Load dữ liệu

MovieLens 1M gồm 3 file, mỗi file dùng `::` làm separator:

In [ ]:
# Định nghĩa columns cho từng file
ratings_cols = ['userId', 'movieId', 'rating', 'timestamp']
movies_cols  = ['movieId', 'title', 'genres']
users_cols  = ['userId', 'gender', 'age', 'occupation', 'zipcode']

# Load ratings
ratings = pd.read_csv(
    'data/raw/ml-1m/ratings.dat',
    sep='::', engine='python', names=ratings_cols
)

# Load movies
movies = pd.read_csv(
    'data/raw/ml-1m/movies.dat',
    sep='::', engine='python', names=movies_cols, encoding='latin-1'
)

# Load users
users = pd.read_csv(
    'data/raw/ml-1m/users.dat',
    sep='::', engine='python', names=users_cols
)

print(f'📊 Ratings:  {len(ratings):,} dòng')
print(f'🎬 Movies:    {len(movies):,} dòng')
print(f'👤 Users:     {len(users):,} dòng')

## 5. Khám phá Ratings

In [ ]:
# Xem 5 dòng đầu
ratings.head()

In [ ]:
# Thông tin cơ bản
print('=== RATINGS ===')
print(ratings.dtypes)
print('\nMissing values:', ratings.isnull().sum().sum())

print('\n=== RATING STATS ===')
print(ratings['rating'].describe())

print('\n=== PHÂN BỐ RATING ===')
print(ratings['rating'].value_counts().sort_index())

In [ ]:
# Vẽ phân bố rating
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
ratings['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Phân bố Rating')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Số lượng')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Histogram
ratings['rating'].hist(bins=5, ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Histogram Rating')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Tần suất')

plt.tight_layout()
plt.savefig('results/charts/01_rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 Đã lưu: results/charts/01_rating_distribution.png')

## 6. Khám phá Movies

In [ ]:
movies.head(10)

In [ ]:
# Tách năm phát hành từ title
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)$')
movies['year'] = pd.to_numeric(movies['year'], errors='coerce')

print(f'Năm phát hành: {movies["year"].min():.0f} – {movies["year"].max():.0f}')

# Đếm số genre trung bình
movies['num_genres'] = movies['genres'].str.count('\|') + 1
movies.loc[movies['genres'] == '(no genres listed)', 'num_genres'] = 0

print(f'Trung bình genres/phim: {movies["num_genres"].mean():.2f}')

# Top genres
all_genres = movies['genres'].str.split('|').explode()
print('\nTop 10 Genres:')
print(all_genres.value_counts().head(10))

In [ ]:
# Vẽ phim theo năm và top genres
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Phim theo năm
year_counts = movies.dropna(subset=['year']).groupby('year').size()
year_counts.plot(ax=axes[0], color='steelblue', linewidth=1)
axes[0].set_title('Số phim theo năm phát hành')
axes[0].set_xlabel('Năm')
axes[0].set_ylabel('Số phim')

# Top genres
all_genres.value_counts().head(10).plot(
    kind='barh', ax=axes[1], color='steelblue', edgecolor='black'
)
axes[1].set_title('Top 10 Genres')
axes[1].set_xlabel('Số phim')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('results/charts/01_movies_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Khám phá Users

In [ ]:
users.head()

In [ ]:
# Map age groups
age_map = {
    1: 'Under 18', 18: '18-24', 25: '25-34', 35: '35-44',
    45: '45-49', 50: '50-55', 56: '56+'
}
users['age_group'] = users['age'].map(age_map)

# Map occupations
occ_map = {
    0:'other',1:'academic',2:'artist',3:'clerical',4:'student',
    5:'customer service',6:'doctor',7:'executive',8:'farmer',
    9:'homemaker',10:'K-12 student',11:'lawyer',12:'programmer',
    13:'retired',14:'sales',15:'scientist',16:'self-employed',
    17:'engineer',18:'tradesman',19:'unemployed',20:'writer'
}
users['occupation_name'] = users['occupation'].map(occ_map)

print('Gender distribution:')
print(users['gender'].value_counts())

print('\nAge group distribution:')
print(users['age_group'].value_counts())

print('\nTop occupations:')
print(users['occupation_name'].value_counts().head(10))

In [ ]:
# Vẽ users demographics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

users['gender'].value_counts().plot(
    kind='bar', ax=axes[0], color=['coral','steelblue'], edgecolor='black'
)
axes[0].set_title('Gender')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

users['age_group'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='steelblue', edgecolor='black'
)
axes[1].set_title('Age Group')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

users['occupation_name'].value_counts().head(10).plot(
    kind='barh', ax=axes[2], color='steelblue', edgecolor='black'
)
axes[2].set_title('Top 10 Occupations')
axes[2].set_xlabel('Số users')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('results/charts/01_users_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Ratings per User & per Movie

In [ ]:
# Số ratings mỗi user
ratings_per_user = ratings.groupby('userId').size()
print('Ratings per User:')
print(f'  Min:    {ratings_per_user.min()}')
print(f'  Max:    {ratings_per_user.max()}')
print(f'  Mean:   {ratings_per_user.mean():.2f}')
print(f'  Median: {ratings_per_user.median():.0f}')

# Số ratings mỗi movie
ratings_per_movie = ratings.groupby('movieId').size()
print('\nRatings per Movie:')
print(f'  Min:    {ratings_per_movie.min()}')
print(f'  Max:    {ratings_per_movie.max()}')
print(f'  Mean:   {ratings_per_movie.mean():.2f}')
print(f'  Median: {ratings_per_movie.median():.0f}')

In [ ]:
# Sparse matrix check
n_users = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
total_possible = n_users * n_movies
sparsity = 1 - (len(ratings) / total_possible)

print(f'Tổng số users: {n_users:,}')
print(f'Tổng số movies: {n_movies:,}')
print(f'Tổng số ratings: {len(ratings):,}')
print(f'Sparsity (độ thưa): {sparsity*100:.2f}%')
print('→ Tức là chỉ có {:.2f}% cells có giá trị!'.format((1-sparsity)*100))

In [ ]:
# Vẽ ratings per user/movie
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ratings_per_user.hist(bins=50, ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Phân bố Ratings/User')
axes[0].set_xlabel('Số ratings')
axes[0].set_ylabel('Số users')
axes[0].axvline(ratings_per_user.mean(), color='red', linestyle='--', label=f'Mean: {ratings_per_user.mean():.1f}')
axes[0].legend()

ratings_per_movie.hist(bins=50, ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Phân bố Ratings/Movie')
axes[1].set_xlabel('Số ratings')
axes[1].set_ylabel('Số movies')
axes[1].axvline(ratings_per_movie.mean(), color='red', linestyle='--', label=f'Mean: {ratings_per_movie.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/charts/01_ratings_per_user_movie.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Tổng kết EDA

| Metric | Giá trị |
|--------|---------|
| Tổng ratings | 1,000,209 |
| Tổng users | 6,040 |
| Tổng movies | 3,706 |
| Rating scale | 1–5 ★ |
| Sparsity | 95.53% |
| Avg ratings/user | 165.6 |
| Avg ratings/movie | 269.9 |

**Nhận xét:** Dataset rất thưa (95%+ empty cells) — đây là lý do tại sao cần thuật toán gợi ý thông minh thay vì chỉ dùng basic statistics!

In [ ]:
# Lưu cleaned data cho notebook tiếp theo
import os
os.makedirs('data/processed', exist_ok=True)

ratings.to_csv('data/processed/ratings_clean.csv', index=False)
movies.to_csv('data/processed/movies_clean.csv', index=False)
users.to_csv('data/processed/users_clean.csv', index=False)

print('✅ Đã lưu cleaned data vào data/processed/')